# 7.5. Pooling
D2L의 Pooling장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Pooling이란?

Convolution을 거치면 이미지가 여러 개의 Feature Map으로 변한다.

예를 들어서 

```text
입력이미지
[32, 3, 64, 64]

↓ Conv2d

[32, 16, 64, 64]
```

하지만 CNN이 깊어질수록 계속 64 x 64 크기를 유지하면

- 계산량이 커지고
- 메모리를 많이 사용하고
- 너무 세밀한 위치 정보에 민감해질 수 있다.

Pooling은 Feature Map의 중요한 정보는 유지하면서 공간 크기 H x W를 줄이는 연산이다.

Pooling은 물체나 특징이 몇 픽셀 정도 이동해도 비슷한 특징으로 인식하도록 만드는 효과도 있습니다. D2L에서는 이를 위치 변화에 대한 민감도를 낮추고 spatial downsampling을 수행하는 두 가지 목적으로 설명한다.

## 2. Pooling의 기본 원리

Pooling도 convolution처럼 작은 window를 이미지 위에서 이동시킨다.

예를 들어서 2 x 2 Pooling Window라면

```text
1 2
3 4
```

이 4개의 값을 하나로 압축한다. 어떤 방법으로 압축하느냐에 따라 대표적으로

1. Max Pooling
2. Average Pooling

두 종류가 있다.

## 3. Max Pooling

가장 많이 쓰이는게 Max Pooling이다. Pooling window 내부에 가장 큰 값 하나만 남긴다.

```text
1  5
2  3
```
이라면

max(1, 5, 2, 3) = 5

따라서 출력은 5가 된다.

```text
1  2  3  4
5  6  7  8
9 10 11 12
13 14 15 16

여기에서 2 x 2, stride = 2 Max Pooling을 쓰면

[1  2]  [3  4]
[5  6]  [7  8]

[9 10]  [11 12]
[13 14] [15 16]
```
각 영역의 최대값을 가져온다.

## 4. Average Pooling

Average Pooling은 Pooling Window 안에 있는 값들의 평균을 사용한다.

```text
1  5
2  4
```
이라면

(1 + 5 + 2 + 4) / 4 = 3

따라서 출력은 3이 된다.

## 5. Pooling에는 학습되는 Parameter가 없다.

Convolution에서는 Kernel의 Weight가 학습된다. 하지만 Pooling에는 Weight가 없다.

Max Pooling: max(영역)
Average Pooling: mean(영역)

을 계산할 뿐이다.

따라서 Pooling Layer 자체에는 학습할 Parameter가 없다.

## 6. 직접 Max Pooling 구현

In [2]:
X = torch.tensor([
    [0., 1., 2.],
    [3., 4., 5.],
    [6., 7., 8.]
])

X

tensor([[0., 1., 2.],
        [3., 4., 5.],
        [6., 7., 8.]])

In [3]:
X[0:2, 0:2]

tensor([[0., 1.],
        [3., 4.]])

In [4]:
X[0:2, 0:2].max()

tensor(4.)

Max Pooling은 이 작업을 Feature Map 돌아다니며 반복하는 것이다.

## 7. PyTorch의 MaxPool2d

In [6]:
X = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)

print(X)
print(X.shape)

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])
torch.Size([1, 1, 4, 4])


In [7]:
pool = nn.MaxPool2d(
    kernel_size=2,
    stride=2
)

Y = pool(X)

print(Y)
print(Y.shape)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])
torch.Size([1, 1, 2, 2])


## 8. Pooling의 Stride

Stride는 Pooling Window가 한 번에 몇 칸씩 이동하는지를 의미한다.

Pooling에서 흔히
    nn.MaxPool2d(kernel_size=2, stride=2)
이걸 많이 쓴다. 그러면 공간 크기가 절반씩 줄어든다.

PyTorch에서 stride를 생략하면 기본적으로 kernel_size와 같은 값이 사용된다.

## 9. Pooling 출력 크기 계산

Pooling 출력 크기는 다음과 같이 계산할 수 있다.

$$
H_{out}
=
\left\lfloor
\frac{H_{in} + 2P - K}{S}
\right\rfloor
+ 1
$$

여기서

- H_in : 입력 크기
- K : Pooling Window 크기
- P : Padding
- S : Stride

## 10. Pooling과 Channel

Pooling은 Channel수를 바꾸지 않는다. Pooling은 각각의 Channel에 독립적으로 적용된다.

Conv2d
채널 수를 바꿀 수 있음

Pooling
채널 수 안 바꿈
H, W만 줄임

Pooling은 convolution처럼 여러 input channel을 합산하지 않는다. 각 feature channel에 따로 pooling을 수행하므로 입력 채널 수와 출력 채널 수가 같다.

## 11. Conv와 Pooling

### Convolution

어떤 특징이 있는지 찾는다.

### Pooling

찾은 특징의 중요한 정보만 남기고 크기를 줄인다.

## 12. 오늘의 정리

- Pooling은 Feature Map의 공간 크기 H × W를 줄이는 연산이다.
- Pooling에는 학습되는 Weight나 Bias가 없다.
- Max Pooling은 영역에서 가장 큰 값을 선택한다.
- Average Pooling은 영역의 평균값을 사용한다.
- CNN에서는 Max Pooling이 많이 사용된다.
- Pooling의 kernel_size는 한 번에 확인할 영역의 크기이다.
- Pooling의 stride는 Pooling Window가 이동하는 간격이다.
- `MaxPool2d(2, 2)`를 사용하면 보통 H와 W가 절반으로 감소한다.
- Pooling은 각 Channel에 독립적으로 적용된다.
- 따라서 Pooling은 Channel 수를 변경하지 않는다.
- Convolution은 특징을 학습해서 찾고, Pooling은 찾은 특징을 압축한다.